In [ ]:
import cv2
import numpy as np

widthImg = 640
heightImg = 480

cap = cv2.VideoCapture(0)
cap.set(3, widthImg)
cap.set(4, heightImg)
cap.set(10, 150) 

def preProcessing(img):
    
    imgGray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    imgBlur = cv2.GaussianBlur(imgGray, (5, 5), 1)
    imgThres = cv2.Canny(imgBlur, 200, 200)
    
    kernel = np.ones((5, 5))
    imgDial = cv2.dilate(imgThres, kernel, iterations=2)
    imgThreshold = cv2.erode(imgDial, kernel, iterations=1)
    
    imgContours = img.copy()
    contours, hierarchy = cv2.findContours(imgThreshold, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(imgContours, contours, -1, (0, 255, 0), 2)
    
    return imgGray, imgBlur, imgThreshold, imgContours, contours

def reorder(myPoints):
    
    myPoints = myPoints.reshape((4, 2))
    myPointsNew = np.zeros((4, 1, 2), np.int32)
    add = myPoints.sum(1)
    myPointsNew[0] = myPoints[np.argmin(add)]
    myPointsNew[3] = myPoints[np.argmax(add)]
    diff = np.diff(myPoints, axis=1)
    myPointsNew[1] = myPoints[np.argmin(diff)]
    myPointsNew[2] = myPoints[np.argmax(diff)]
    return myPointsNew

def warpAndEnhance(img, contours):
    
    imgBigContour = img.copy()
    biggest = np.array([])
    maxArea = 0
    
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area > 5000:
            peri = cv2.arcLength(cnt, True)
            approx = cv2.approxPolyDP(cnt, 0.02 * peri, True)
            if area > maxArea and len(approx) == 4:
                biggest = approx
                maxArea = area
                
    if biggest.size == 0:
        biggest = np.array([[0, 0], [widthImg, 0], [0, heightImg], [widthImg, heightImg]])
        
    cv2.drawContours(imgBigContour, biggest, -1, (0, 255, 0), 10)
    
    biggestReordered = reorder(biggest)
    pts1 = np.float32(biggestReordered)
    pts2 = np.float32([[0, 0], [widthImg, 0], [0, heightImg], [widthImg, heightImg]])
    matrix = cv2.getPerspectiveTransform(pts1, pts2)
    imgWarpColor = cv2.warpPerspective(img, matrix, (widthImg, heightImg))
    
    imgWarpGray = cv2.cvtColor(imgWarpColor, cv2.COLOR_BGR2GRAY)
    imgAdaptiveThres = cv2.adaptiveThreshold(imgWarpGray, 255, 1, 1, 11, 2)
    imgAdaptiveThres = cv2.bitwise_not(imgAdaptiveThres)
    imgAdaptiveThres = cv2.medianBlur(imgAdaptiveThres, 3)
    
    return imgBigContour, imgWarpColor, imgWarpGray, imgAdaptiveThres

def stackImages(scale, imgArray):
    
    rows = len(imgArray)
    cols = len(imgArray[0])
    rowsAvailable = isinstance(imgArray[0], list)
    width = imgArray[0][0].shape[1]
    height = imgArray[0][0].shape[0]
    if rowsAvailable:
        for x in range(0, rows):
            for y in range(0, cols):
                if imgArray[x][y].shape[:2] == imgArray[0][0].shape[:2]:
                    imgArray[x][y] = cv2.resize(imgArray[x][y], (0, 0), fx=scale, fy=scale)
                else:
                    imgArray[x][y] = cv2.resize(imgArray[x][y], (imgArray[0][0].shape[1], imgArray[0][0].shape[0]), fx=scale, fy=scale)
                if len(imgArray[x][y].shape) == 2: 
                    imgArray[x][y] = cv2.cvtColor(imgArray[x][y], cv2.COLOR_GRAY2BGR)
        imageBlank = np.zeros((height, width, 3), np.uint8)
        hor = [imageBlank] * rows
        for x in range(0, rows):
            hor[x] = np.hstack(imgArray[x])
        ver = np.vstack(hor)
    else:
        for x in range(0, rows):
            if imgArray[x].shape[:2] == imgArray[0].shape[:2]:
                imgArray[x] = cv2.resize(imgArray[x], (0, 0), fx=scale, fy=scale)
            else:
                imgArray[x] = cv2.resize(imgArray[x], (imgArray[0].shape[1], imgArray[0].shape[0]), fx=scale, fy=scale)
            if len(imgArray[x].shape) == 2: 
                imgArray[x] = cv2.cvtColor(imgArray[x], cv2.COLOR_GRAY2BGR)
        hor = np.hstack(imgArray)
        ver = hor
    return ver

try:
    while True:
        success, img = cap.read()
        if not success:
            print("Failed to Capture Webcam Frames")
            break
            
        imgGray, imgBlur, imgThreshold, imgContours, contours = preProcessing(img)
        imgBigContour, imgWarpColor, imgWarpGray, imgAdaptiveThres = warpAndEnhance(img, contours)
        imageArray = [
            [img, imgGray, imgThreshold, imgContours],
            [imgBigContour, imgWarpColor, imgWarpGray, imgAdaptiveThres]
        ]
        
        stackedImage = stackImages(0.5, imageArray)
        cv2.imshow("Document Scanner Real-time Matrix Pipeline", stackedImage)
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

except BaseException as e:
    print(f"An Error has Occured : {e}")